# 05 — Compute Neural Regression Collapse (NRC) Geometry

**The formula gate, cleared.** Every formula in `src/geometry/nrc.py` is
transcribed exactly from:

> Andriopoulos, Dong, Guo, Zhao & Ross, *The Prevalence of Neural Collapse
> in Neural Multivariate Regression*, NeurIPS 2024 (arXiv:2409.04180) —
> Section 3.1 (NRC1/NRC2/NRC3 definitions) and Appendix F, Theorem F.1
> (closed-form optimal gamma for NRC3).

Fetched and read in full (the NeurIPS proceedings PDF, not a summary)
before a single line of `nrc.py` was written.

**A finding from actually reading the paper that changes this pilot's
scope:** the paper states directly (Appendix A.3) that **NRC3 is trivially
zero and not meaningful for univariate regression (n=1)** — the exact
case of every dataset in the `uci` pilot group (all scalar targets). This
notebook therefore computes **NRC1 and NRC2 only**; `compute_nrc3` returns
`None` for n=1 by design, not by omission. A meaningful NRC3 would need a
multivariate-target dataset (n>1), which is out of scope for this pilot
group — see the "trục n" (target-dimension axis) direction in the
project's research proposal for where that belongs.

**Verified against the paper's own theory, not just this module's own
logic:** `tests/test_nrc.py` includes a test that constructs H, W, Y using
the paper's own Theorem 4.1 / Corollary 4.2 closed-form optimal solution
(worked by hand for the n=2, uncorrelated-target case) and checks that
NRC1, NRC2, NRC3 as implemented here all correctly evaluate to ~0 on it —
an independent check against the source math, not just this code checking
itself. 21/21 tests pass.

**Architecture bridge, explicit:** this project's models are mixture-density
networks (03/04), not the plain single-linear-head regressors the NRC
papers study. `extract_mean_head_weight` slices out only the mean-prediction
rows of the output layer as "W" in the NRC sense — see that function's
docstring for why the std/mixture-logit rows are excluded.

**Expected runtime:** well under a minute — this is linear algebra on
small feature matrices, no training.
**GPU:** not used (everything here runs on the numpy features saved by `04`).


## Step 1 — Locate project, import helpers

In [ ]:
import sys
from pathlib import Path

if "PROJECT_ROOT" not in dir():
    _here = Path.cwd()
    for candidate in [_here, *_here.parents]:
        if (candidate / "src" / "utils" / "env_utils.py").exists():
            PROJECT_ROOT = candidate
            break
    else:
        raise FileNotFoundError("PROJECT_ROOT not found. Run 00_environment.ipynb first.")

if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from src.geometry import nrc

print(f"PROJECT_ROOT = {PROJECT_ROOT}")
print("src.geometry.nrc imported -- formula gate cleared, see markdown above for sourcing.")


## Step 2 — Load extracted features (`04`) and checkpoints (`03`)

Needs both: features for NRC1/NRC2, and the checkpoint's mean-head weight
matrix for NRC2 (and NRC3, where applicable).

In [ ]:
import pickle
import torch
from src.models import pilot_mixture_model as pmm

MIXTURE_SIZE = 1  # must match 03/04

EXTERNAL_DIR = PROJECT_ROOT / "external" / "quantile-recalibration-training"
if not EXTERNAL_DIR.exists():
    from src.utils import env_utils
    env_utils.clone_or_pull_repo(
        repo_url="https://github.com/Vekteur/quantile-recalibration-training.git",
        dest=EXTERNAL_DIR, branch="main",
    )
MixturePrediction = pmm.import_mixture_prediction(PROJECT_ROOT)
pmm.set_mixture_prediction_cls(MixturePrediction)

outputs_dir = PATHS["outputs"] if "PATHS" in dir() else PROJECT_ROOT / "outputs"
features_path = outputs_dir / f"uci_pilot_features_mixture_{MIXTURE_SIZE}.pkl"
if not features_path.exists():
    raise FileNotFoundError(f"{features_path} not found -- run 04_extract_features.ipynb first.")
with open(features_path, "rb") as f:
    saved = pickle.load(f)
all_features = saved["features"]

ckpt_dir = (PATHS["checkpoints"] if "PATHS" in dir() else PROJECT_ROOT / "checkpoints") / f"mixture_{MIXTURE_SIZE}"
print(f"Loaded features for {len(all_features)} datasets.")


## Step 3 — Compute NRC1/NRC2 per dataset

Uses the **calibration split's** features and targets — the split this
project's diagnostic is meant to be applied to (the held-out set a
deployed, frozen model would be calibrated against), not train (which the
model has directly fit) or test (reserved for the correlation target in
`06`).

In [ ]:
results = {}

for name, per_split in all_features.items():
    ckpt = torch.load(ckpt_dir / f"{name}.pt", map_location="cpu", weights_only=True)
    output_layer_weight = None
    module = pmm.load_pilot_checkpoint(ckpt_dir / f"{name}.pt")
    output_layer_weight = module.model.body.output_layer.weight.detach().numpy()
    W_mean = nrc.extract_mean_head_weight(output_layer_weight, mixture_size=MIXTURE_SIZE)

    H_calib = per_split["calib"]["features"]
    y_calib = per_split["calib"]["y"]
    n_target_dim = y_calib.shape[1]

    nrc1 = nrc.compute_nrc1(H_calib, n_target_dim=n_target_dim)
    nrc2 = nrc.compute_nrc2(H_calib, W_mean)

    nrc3 = None
    if n_target_dim > 1:
        Sigma_sqrt = nrc.compute_target_covariance_sqrt(y_calib)
        nrc3 = nrc.compute_nrc3(W_mean, Sigma_sqrt, n_target_dim=n_target_dim)

    results[name] = {"nrc1": nrc1, "nrc2": nrc2, "nrc3": nrc3, "n_target_dim": n_target_dim,
                      "n_calib_samples": H_calib.shape[0]}
    print(f"  {name:10s}  NRC1={nrc1:.4f}  NRC2={nrc2:.4f}  "
          f"NRC3={'N/A (n=1)' if nrc3 is None else f'{nrc3:.4f}'}  (M_calib={H_calib.shape[0]})")


## Step 4 — Sanity checks

NRC1/NRC2 are squared-norm residuals of unit vectors projected onto a
subspace of the same or smaller dimension — they are mathematically
bounded in [0, 2] (worst case: a unit vector exactly orthogonal to the
subspace has squared residual norm 1, and no case can exceed that bound
by more than a small amount from projection numerics). Values outside a
sane range would indicate a bug, not just "bad" data.

In [ ]:
import math

problems = []
for name, r in results.items():
    for metric in ("nrc1", "nrc2"):
        val = r[metric]
        if not math.isfinite(val):
            problems.append(f"{name}/{metric}: non-finite ({val})")
        elif not (0 <= val <= 2.01):
            problems.append(f"{name}/{metric}: out of expected [0,2] range ({val:.4f})")

if problems:
    print("[warn] Issues found:")
    for p in problems:
        print(f"  - {p}")
else:
    print(f"All {len(results)} datasets: NRC1/NRC2 finite and within the expected [0, 2] range.")


## Step 5 — Save NRC results for `06`

In [ ]:
out_path = outputs_dir / f"uci_pilot_nrc_mixture_{MIXTURE_SIZE}.pkl"
with open(out_path, "wb") as f:
    pickle.dump({"nrc_results": results, "mixture_size": MIXTURE_SIZE}, f)
print(f"Saved to {out_path}")


## A methodological tension worth flagging before `06`

**The synthetic-fixture test run above shows moderate, not strong, NRC1/NRC2
collapse (~0.7-0.8, where 0 would be perfect collapse).** Two candidate
explanations, both worth checking on real data rather than assuming either:

1. **Early stopping vs. "terminal phase" training.** The source NRC paper's
   own experiments train for a very long time *past* convergence (up to
   1.5M epochs for their MuJoCo datasets) specifically to reach neural
   collapse's "terminal phase." This project's training protocol (`03`)
   uses early stopping on validation NLL — the *correct* choice for a
   calibration-quality model, but potentially in tension with NRC theory's
   own experimental regime, which keeps training well past the point
   early stopping would halt it. `AdamW`'s default `weight_decay=0.01` is
   active in `03` (verified, not assumed — `torch.optim.AdamW`'s actual
   default, confirmed by inspection), so nonzero regularization *is*
   present (the paper's Section 3.2/4.4 finding is that collapse requires
   nonzero weight decay, however small) -- but early stopping may simply
   not run long enough for that regularization to fully express itself
   geometrically.
2. It could also simply be that this particular synthetic linear task is
   easy enough that the network doesn't need strongly collapsed features
   to fit it well — real UCI data may behave differently.

**This is worth deciding explicitly, not silently defaulting on:** before
trusting a null/weak result from `06`'s correlation on real data, re-run
`03` for a fixed, much larger number of epochs *without* early stopping
(disable the calibration-motivated stopping criterion, matching the source
paper's own protocol instead) on at least a couple of pilot datasets, and
check whether NRC1/NRC2 drop substantially further with continued
training. If they do, the go/no-go correlation should be computed at (or
compared across) both checkpoints — the early-stopped one (matching
deployment practice) and the extended one (matching NRC theory's own
regime) — since they may tell different stories.


## Next steps — the actual go/no-go

**Next:** `06_correlation_analysis.ipynb` — this is the notebook that
answers the real research question: does NRC-distance (this notebook)
predict PCE (computed fresh in `06`, using the exact PCE/Quantile
Recalibration formulas already read in full from the QRT paper earlier in
this project — no new formula gate needed there, unlike here)? That
correlation is the actual go/no-go decision point for the whole NRC-Cal
direction.
